# 02 — MRMR Channel Selection Analysis

Interactive walkthrough of the Minimum Redundancy Maximum Relevance (MRMR)
channel selection algorithm applied to EMG signals.

This notebook demonstrates:
1. Computing mutual information between each EMG channel and grip force
2. Computing pairwise redundancy between channels
3. Greedy MRMR forward selection
4. Visualizing results

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_NINAPRO_DIR, NINAPRO_FS, NINAPRO_CHANNEL_NAMES,
    ENVELOPE_WINDOW_MS, MRMR_N_SELECT, MRMR_SUBSAMPLE, RANDOM_SEED,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data and Compute RMS Envelopes

In [ ]:
from src.data.load_ninapro import load_all_ninapro
from src.features.mrmr import _compute_rms_envelope

# Load all subjects
subject_data = load_all_ninapro(str(RAW_NINAPRO_DIR))
print(f"Loaded {len(subject_data)} subjects")

# Compute RMS envelopes and concatenate
all_envelopes = []
all_forces = []

for subj in subject_data:
    emg = subj['emg']
    force = subj['force']
    
    rms_env = _compute_rms_envelope(emg, NINAPRO_FS, ENVELOPE_WINDOW_MS)
    
    if force.ndim == 2:
        force = np.sum(force, axis=1)
    
    win_len = int(NINAPRO_FS * ENVELOPE_WINDOW_MS / 1000)
    n_windows = len(rms_env)
    force_ds = force[:n_windows * win_len].reshape(n_windows, win_len).mean(axis=1)
    
    all_envelopes.append(rms_env)
    all_forces.append(force_ds)

X = np.concatenate(all_envelopes, axis=0)
y = np.concatenate(all_forces, axis=0)

print(f"Concatenated data: {X.shape[0]} samples, {X.shape[1]} channels")

## 2. Subsample for Efficiency

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)

if len(X) > MRMR_SUBSAMPLE:
    idx = rng.choice(len(X), size=MRMR_SUBSAMPLE, replace=False)
    idx.sort()
    X_sub = X[idx]
    y_sub = y[idx]
    print(f"Subsampled to {MRMR_SUBSAMPLE} points")
else:
    X_sub, y_sub = X, y
    print(f"Using all {len(X)} points")

## 3. Step 1: Compute Relevance — MI(channel, force)

In [ ]:
from src.features.mrmr import compute_channel_relevance

relevance = compute_channel_relevance(X_sub, y_sub)

# Print sorted by relevance
sorted_idx = np.argsort(relevance)[::-1]
print("Channel Relevance (MI with Force):")
print("-" * 40)
for i in sorted_idx:
    print(f"  {NINAPRO_CHANNEL_NAMES[i]:>25s}: {relevance[i]:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0'] * len(relevance)
colors[sorted_idx[0]] = '#55A868'  # highlight top
ax.bar(range(len(relevance)), relevance, color=colors, alpha=0.85)
ax.set_xticks(range(len(relevance)))
ax.set_xticklabels(NINAPRO_CHANNEL_NAMES, rotation=45, ha='right')
ax.set_ylabel('Mutual Information')
ax.set_title('Step 1: Channel Relevance — MI(channel, force)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Step 2: Compute Pairwise Redundancy

In [ ]:
from src.features.mrmr import compute_redundancy_matrix

print("Computing pairwise MI (this may take a minute)...")
redundancy = compute_redundancy_matrix(X_sub)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    redundancy,
    xticklabels=NINAPRO_CHANNEL_NAMES,
    yticklabels=NINAPRO_CHANNEL_NAMES,
    annot=True, fmt='.3f', cmap='YlOrRd',
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Mutual Information'},
)
ax.set_title('Pairwise Redundancy (MI between EMG Channels)')
plt.tight_layout()
plt.show()

## 5. Step 3: MRMR Greedy Selection

In [ ]:
from src.features.mrmr import mrmr_select

result = mrmr_select(
    X_sub, y_sub,
    n_select=MRMR_N_SELECT,
    channel_names=NINAPRO_CHANNEL_NAMES,
)

print("MRMR Selection Results:")
print("=" * 50)
for step in result['mrmr_scores']:
    print(f"  Step {step['step']}: {step['selected_name']}")
    print(f"    MRMR Score: {step['mrmr_score']:.4f}")
    print(f"    Relevance:  {step['relevance']:.4f}")
    if step['step'] > 1:
        redundancy_penalty = step['relevance'] - step['mrmr_score']
        print(f"    Redundancy penalty: {redundancy_penalty:.4f}")
    print()

print(f"Selected channels: {result['selected_names']}")
print(f"Selected indices:  {result['selected_indices']}")

## 6. MRMR Score Decomposition

In [ ]:
# Compare relevance vs final MRMR score for all channels
mrmr_scores = result['all_channel_mrmr_scores']
selected = result['selected_indices']

x_pos = np.arange(len(relevance))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x_pos - width/2, relevance, width, label='Relevance (MI with Force)', color='#4C72B0', alpha=0.8)
ax.bar(x_pos + width/2, mrmr_scores, width, label='MRMR Score', color='#DD8452', alpha=0.8)

for idx in selected:
    ax.axvspan(idx - 0.5, idx + 0.5, alpha=0.15, color='green')
    ax.annotate('SELECTED', (idx, max(relevance[idx], mrmr_scores[idx])),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8, fontweight='bold', color='green')

ax.set_xlabel('EMG Channel')
ax.set_ylabel('Score')
ax.set_title('MRMR Channel Selection — Relevance vs MRMR Score')
ax.set_xticks(x_pos)
ax.set_xticklabels(NINAPRO_CHANNEL_NAMES, rotation=45, ha='right', fontsize=9)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Interpretation

The MRMR algorithm selected the channels that provide the best combination of:
- **High relevance**: Strong mutual information with grip force
- **Low redundancy**: Complementary information (not duplicating each other)

Anatomically, we expect the selected channels to correspond to muscles directly
involved in grip force generation (e.g., flexor digitorum, extensor digitorum)
rather than channels that share similar activation patterns.